# Statistics for Data Analysts/Engineers — Module 11: Correlation and Covariance

So far we've mostly asked "do the means/proportions differ?". Now we'll ask a different
question: **do two continuous variables move together?** When a customer's spending
rises, does their satisfaction rise too? Correlation and covariance are the tools for
measuring the strength and direction of such a relationship -- and the foundation for the
linear regression in Module 12.

## Table of contents
1. [Covariance](#sec1)
2. [Pearson correlation](#sec2)
3. [Statistical significance of correlation](#sec3)
4. [Spearman correlation](#sec4)
5. [Correlation is not causation](#sec5)
6. [Correlation matrix and heatmap](#sec6)
7. [Summary and exercises](#sec7)


Every module in this course starts by generating the same synthetic dataset -- thanks to a fixed random seed (`np.random.default_rng(42)`), the numbers always come out identical, so you can open any module independently of the others, even after restarting the kernel.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 400

city = rng.choice(
    ["New York", "Chicago", "Austin", "Seattle", "Denver"],
    size=n, p=[0.30, 0.25, 0.15, 0.15, 0.15],
)
region = rng.choice(["North", "South", "East", "West"], size=n)
age = rng.integers(18, 70, size=n)

# monthly spend: right-skewed distribution (typical for spending data)
monthly_spend = rng.lognormal(mean=5.2, sigma=0.5, size=n).round(2)

# satisfaction correlated with spend + a region effect (deliberately different) + random
# noise, clipped to a 1-10 scale -- this gives Module 10 (ANOVA) a real between-group gap
region_effect = {"North": 0.0, "South": -0.5, "East": 0.3, "West": 0.8}
satisfaction = np.clip(
    monthly_spend / 45
    + np.array([region_effect[r] for r in region])
    + rng.normal(0, 1.5, size=n),
    1, 10,
).round(1)

# A/B campaign group -- with a REAL injected effect on conversion (group B better)
group = rng.choice(["A", "B"], size=n)
conversion_prob = np.where(group == "A", 0.10, 0.20)
converted = rng.binomial(1, conversion_prob)

# spend before/after a loyalty-program rollout (paired data)
spend_before = rng.normal(220, 40, size=n).round(2)
program_effect = rng.normal(15, 10, size=n)
spend_after = (spend_before + program_effect).round(2)

customers = pd.DataFrame({
    "customer_id": np.arange(1, n + 1),
    "city": city,
    "region": region,
    "age": age,
    "monthly_spend": monthly_spend,
    "satisfaction": satisfaction,
    "campaign_group": group,
    "converted": converted,
    "spend_before": spend_before,
    "spend_after": spend_after,
})
customers.head()


<a id="sec1"></a>
## 1. Covariance

**Covariance** measures whether two variables move in the same direction (positive
covariance: when one rises, the other usually rises too), in opposite directions
(negative), or independently (close to zero):

```
cov(X, Y) = mean of [(x - mean_x) * (y - mean_y)]
```

In [ ]:
covariance = customers["monthly_spend"].cov(customers["satisfaction"])
print(f"Covariance (spend, satisfaction): {covariance:.2f}")


> ⚠️ **Covariance is practically useless on its own**
>
> The value of covariance depends on the UNITS of both variables -- if we expressed spend in cents instead of dollars, covariance would grow a hundredfold, even though the "strength" of the relationship hasn't changed at all. Covariance only tells you the DIRECTION of the relationship (positive/negative), never its STRENGTH in a way that's comparable across different pairs of variables. That's why in practice almost everyone uses correlation (section 2) instead -- a normalized version of covariance.

<a id="sec2"></a>
## 2. Pearson correlation

**Pearson correlation** (`r`) is covariance divided by the product of both variables'
standard deviations -- which keeps it always within the range `[-1, 1]`, regardless of
units:

```
r = cov(X, Y) / (std_x * std_y)
```

- `r = 1` -- a perfect positive LINEAR relationship,
- `r = -1` -- a perfect negative linear relationship,
- `r = 0` -- no LINEAR relationship (careful: this doesn't mean "no relationship at
  all"!),
- intermediate values -- partial linear relationship, the closer to `1`/`-1`, the
  stronger.

In [ ]:
from scipy import stats

r, p_value = stats.pearsonr(customers["monthly_spend"], customers["satisfaction"])
print(f"Pearson r: {r:.3f}")
print(f"p-value: {p_value:.2e}")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 5))
plt.scatter(customers["monthly_spend"], customers["satisfaction"], alpha=0.5)
plt.xlabel("Monthly spend")
plt.ylabel("Satisfaction")
plt.title(f"r = {r:.3f}")
plt.show()


> 📊 **Anscombe knocks again**
>
> Remember Anscombe's quartet from Module 2? Same problem here: `r` measures ONLY a LINEAR relationship. A strong curved relationship (e.g. parabolic) can give an `r` close to zero, despite a clear, predictable pattern. Always draw a scatter plot -- exactly like we did above -- before trusting the `r` number alone.

<a id="sec3"></a>
## 3. Statistical significance of correlation

`stats.pearsonr` returns not just `r`, but also a p-value -- a test of whether `r`
differs significantly from zero in the population:

- H0: `rho = 0` (no linear correlation in the population)
- H1: `rho != 0`

In [ ]:
alpha = 0.05
print(f"r={r:.3f}, p={p_value:.2e}")
print("Correlation is statistically significant" if p_value <= alpha else "Correlation is NOT statistically significant")


> ⚠️ **Statistical significance is not the same as strength of relationship**
>
> Exactly like in Module 7: with a large sample, even a VERY WEAK correlation (e.g. `r=0.05`) can come out "statistically significant". Always look at the VALUE of `r` (strength of relationship), not just the p-value (whether it differs from zero). Our spend-satisfaction correlation is both strong AND significant -- but that doesn't always go hand in hand.

<a id="sec4"></a>
## 4. Spearman correlation

**Spearman correlation** computes Pearson correlation not on the raw values, but on
their RANKS (position in sorted order). This lets it detect any **monotonic**
relationship (increasing or decreasing), not necessarily linear -- and it's much more
robust to outliers.

In [ ]:
import numpy as np

# example: strongly NONlinear, but monotonic relationship
rng = np.random.default_rng(1)
x_example = np.linspace(1, 10, 100)
y_example = x_example ** 3 + rng.normal(0, 20, size=100)  # cubic relationship

r_pearson_example, _ = stats.pearsonr(x_example, y_example)
r_spearman_example, _ = stats.spearmanr(x_example, y_example)

print(f"Pearson r: {r_pearson_example:.3f}")
print(f"Spearman r: {r_spearman_example:.3f}")

plt.figure(figsize=(7, 5))
plt.scatter(x_example, y_example, alpha=0.5)
plt.title(f"Pearson={r_pearson_example:.3f}, Spearman={r_spearman_example:.3f}")
plt.show()


Spearman should come out clearly higher than Pearson -- because the CUBIC relationship
(`x^3`) is strongly monotonic (bigger `x` always means bigger `y`, no exceptions), but
not linear, so Pearson won't fully credit it.

In [ ]:
r_spearman_data, p_spearman_data = stats.spearmanr(customers["monthly_spend"], customers["satisfaction"])
print(f"Spearman (spend, satisfaction): r={r_spearman_data:.3f}, p={p_spearman_data:.2e}")
print(f"(for comparison, Pearson: r={r:.3f})")


<a id="sec5"></a>
## 5. Correlation is not causation

This might be the single most important sentence in the whole module: **correlation
does not mean causation**. Even a strong, statistically significant correlation between
X and Y doesn't tell you that X CAUSES Y. Possible explanations for an observed
correlation:

1. **X causes Y** (what we often assume, but rarely can prove from correlation alone),
2. **Y causes X** (the reverse direction -- correlation alone doesn't tell you which
   way),
3. **A hidden variable Z causes both** (a *confounding variable*) -- e.g. spend might
   "cause" satisfaction, OR both might result from yet another customer trait (e.g.
   wealth),
4. **Pure coincidence** -- with enough variables compared, some random correlations will
   appear "for free" (see the famous, tongue-in-cheek "spurious correlations" examples --
   e.g. per-capita cheese consumption correlates with the number of people who died
   tangled in their bedsheets).

> 🔥 **The one reliable path to causation: an experiment**
>
> To move from correlation to causation, you generally need a **randomized-assignment experiment** (exactly like our A/B test from Module 7/9!) -- by randomly assigning customers to groups, you eliminate the influence of hidden variables, because both groups should have a similar distribution of every other trait. That's why randomized A/B tests are so valued in practice: they give you causation, not just correlation.

<a id="sec6"></a>
## 6. Correlation matrix and heatmap

When there are more than two numeric variables, it's convenient to compute ALL pairwise
correlations at once -- `DataFrame.corr()` does this in one line, and a heatmap lets you
spot patterns quickly.

In [ ]:
import seaborn as sns

numeric_columns = ["age", "monthly_spend", "satisfaction", "spend_before", "spend_after"]
correlation_matrix = customers[numeric_columns].corr()

plt.figure(figsize=(7, 6))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation matrix")
plt.show()


Notice the very high correlation between `spend_before` and `spend_after` -- that's no
accident, it's a direct consequence of how we built this data back in Module 8
(`spend_after = spend_before + random_effect`): whoever spent a lot before usually
spends a lot after too.

<a id="sec7"></a>
## 7. Summary and exercises

In this module we covered:
- covariance and why it's of limited use on its own (it depends on units),
- Pearson correlation (`r`) as a normalized, interpretable version of covariance,
- the statistical significance of correlation and the trap of confusing it with
  strength of relationship,
- Spearman correlation for monotonic, nonlinear relationships,
- why correlation does NOT mean causation (and four possible explanations),
- a correlation matrix and heatmap for many variables at once.

> 📝 **Exercise 1: Correlation between age and spend**
>
> Compute the Pearson correlation between `age` and `monthly_spend`, along with its p-value. Is it strong, weak, statistically significant?

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
r_age, p_age = stats.pearsonr(customers["age"], customers["monthly_spend"])
print(f"r={r_age:.3f}, p={p_age:.4f}")
```

Age is drawn completely independently of spend in our data, so expect an r close to 0 and no significance.
</details>

> 📝 **Exercise 2: Spearman for satisfaction and spend before**
>
> Compute the Spearman correlation between `spend_before` and `satisfaction`. Compare it with the Pearson correlation for the same pair -- do they differ meaningfully?

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
r_pearson_ex2, _ = stats.pearsonr(customers["spend_before"], customers["satisfaction"])
r_spearman_ex2, _ = stats.spearmanr(customers["spend_before"], customers["satisfaction"])
print(f"Pearson: {r_pearson_ex2:.3f}, Spearman: {r_spearman_ex2:.3f}")
```
</details>

> 📝 **Exercise 3: Correlation with an outlier**
>
> Take the `monthly_spend` and `satisfaction` columns, but add an ARTIFICIAL outlier point (e.g. a customer with spend=5000 and satisfaction=1) to a copied DataFrame. Compute Pearson's r and Spearman's r BEFORE and AFTER adding that point -- which of the two measures changes more?

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
copy_df = customers.copy()
new_row = pd.DataFrame({"monthly_spend": [5000], "satisfaction": [1]})
with_outlier = pd.concat([copy_df[["monthly_spend", "satisfaction"]], new_row], ignore_index=True)

r_before, _ = stats.pearsonr(copy_df["monthly_spend"], copy_df["satisfaction"])
r_after, _ = stats.pearsonr(with_outlier["monthly_spend"], with_outlier["satisfaction"])
rs_before, _ = stats.spearmanr(copy_df["monthly_spend"], copy_df["satisfaction"])
rs_after, _ = stats.spearmanr(with_outlier["monthly_spend"], with_outlier["satisfaction"])

print(f"Pearson: {r_before:.3f} -> {r_after:.3f}")
print(f"Spearman: {rs_before:.3f} -> {rs_after:.3f}")
```

Pearson usually changes much more -- it's based on raw values, so a single extreme point can 'drag' it significantly. Spearman, based on ranks, is much more robust to individual outlying observations.
</details>

> 📝 **Exercise 4: Full correlation matrix with p-values**
>
> Write a function `p_value_matrix(df, columns)` that returns a DataFrame with the p-values of the Pearson correlation for every pair of columns (hint: a nested loop over `columns`, `stats.pearsonr` for each pair). Run it for the same columns as in section 6.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
def p_value_matrix(df, columns):
    result = pd.DataFrame(index=columns, columns=columns, dtype=float)
    for c1 in columns:
        for c2 in columns:
            _, p = stats.pearsonr(df[c1], df[c2])
            result.loc[c1, c2] = p
    return result


p_value_matrix(customers, numeric_columns)
```
</details>

> 🔥 **Challenge: a spurious correlation via a hidden variable**
>
> Build two ARTIFICIAL variables, both dependent on the same third, hidden variable `Z` (e.g. `Z = rng.normal(0, 1, size=500)`, `X = Z + noise1`, `Y = Z + noise2`, where `noise1`/`noise2` are independent random noise). Compute the correlation between `X` and `Y` -- it should come out clearly different from zero, even though `X` has no causal effect on `Y` whatsoever (both are only effects of `Z`). This is a hands-on example of point 3 from section 5.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
rng2 = np.random.default_rng(9)
z_hidden = rng2.normal(0, 1, size=500)
x_fake = z_hidden + rng2.normal(0, 0.5, size=500)
y_fake = z_hidden + rng2.normal(0, 0.5, size=500)

r_fake, p_fake = stats.pearsonr(x_fake, y_fake)
print(f"r(X, Y) = {r_fake:.3f}, p = {p_fake:.2e}")
print("X has no causal effect on Y -- the entire correlation comes from both depending on Z")
```
</details>

## What's next?

Correlation tells you a relationship exists and how strong it is. **Linear regression**
in **Module 12** will go one step further: it will build a mathematical model that lets
you PREDICT the value of one variable from another (or several others), with a concrete
equation and a measure of how well the model fits the data.